In [1]:
import numpy as np
import json
import random
import pickle
from keras.models import load_model
import nltk
from nltk import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()


In [2]:
intents = json.loads(open('intents.json').read())
words = pickle.load(open('words.pkl','rb'))
classes = pickle.load(open('classes.pkl','rb'))
model = load_model('chatbot_model.h5')

In [3]:
def clean_up_sentence(sentence):
    sentece_words = nltk.wordpunct_tokenize(sentence)
    sentece_words = [lemmatizer.lemmatize(word.lower())for word in sentece_words]
    return sentece_words

def bag_of_word(sentence,words):
    sentence_words = clean_up_sentence(sentence)
    bag = [0]*len(words)
    for s in sentence_words:
        for i,word in enumerate(words):
            if word == s:
                bag[i]=1
    return (np.array(bag))

def predict_class(sentence):
    p = bag_of_word(sentence,words)
    p = np.array([p])
    res = model.predict(p)[0]
    result = [[i,r]for i,r in enumerate(res) if r>0.25]
    result.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for i in result:
        return_list.append({'intents':classes[i[0]],'pro':str(i[1])})
    return return_list

def geresponce(ints,json_intents):
    tag = ints[0]['intents']
    intents = json_intents['intents']
    for i in intents:
        if (i['tag'] == tag):
            res = random.choice(i['responses'])
            break
    return res

In [1]:
import gradio as gr

def yes_man(message, history):
    try:
        ints = predict_class(message)
        res = geresponce(ints,intents)
        return res
    except Exception as e:
        return f"Error:{str(e)}"
with gr.Blocks(theme=gr.themes.Soft(), css="""
body {background: #087c65;}
""") as demo:
    yes_man
    gr.Markdown("""
    # 
    <span style="color:gray">   </span>
    """)
    gr.ChatInterface(
        yes_man,
        chatbot=gr.Chatbot(height=500,scale=30,container=False),
        textbox=gr.Textbox(placeholder="Ask me a yes or no question", container=False, scale=10),
        title="🤖 my dokcter",
        description="Ask Yes Man any question",
        examples=["Hello", "Am I cool?", "Are tomatoes vegetables?"],
        cache_examples=True,
)

demo.launch(share=True)

C:\Users\Alby\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Alby\AppData\Local\Temp\ipykernel_9476\2492777297.py:10: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="""


* Running on local URL:  http://127.0.0.1:7860
Using cache from 'c:\Users\Alby\Desktop\py\chat bot\chat bot\.gradio\cached_examples\18' directory. If method or examples have changed since last caching, delete this folder to clear cache.


Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
